# Bjerknes feedback changes over time

## imports

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import seaborn as sns
import xarray as xr
import tqdm
import pathlib
import cmocean
import os
import copy
import time

# Import custom modules
import src.utils

## set plotting specs
sns.set(rc={"axes.facecolor": "white", "axes.grid": False})
sns.set_palette("colorblind")

## bump up DPI
mpl.rcParams["figure.dpi"] = 100

## get filepaths
DATA_FP = pathlib.Path(os.environ["DATA_FP"])
SAVE_FP = pathlib.Path(os.environ["SAVE_FP"])

## Specify constants

In [ ]:
## should we subtract median?
SUBTRACT_MEDIAN = False

## T-variable
T_VAR = "T_3"

## Funcs

### Misc

In [ ]:
def window(x, subtract_median=True):
    """get data in windows"""
    x_windowed = src.utils.get_windowed(x, stride=120)

    ## handle median case
    if subtract_median:
        median = x_windowed.groupby("time.month").median(["time", "member"])
        x_windowed = x_windowed.groupby("time.month") - median

    return x_windowed


def load_consolidated_wide():
    """utility function to load consolidated data"""

    ## directory with data
    CONS_DIR = pathlib.Path(os.environ["DATA_FP"], "cesm", "consolidated_05")

    ## function to align and open
    kwargs = dict(pop_vars=["u", "u_comp", "T", "T_comp", "w", "w_comp"])
    align_and_open = lambda fp: src.utils.align_pop_times(xr.open_dataset(fp), **kwargs)

    ## open data and align pop times
    kwargs = dict(pop_vars=["u", "u_comp", "T", "T_comp", "w", "w_comp"])
    forced = align_and_open(CONS_DIR / "forced.nc")
    anom = align_and_open(CONS_DIR / "anom.nc")

    return forced, anom



def frac_change(x):
    """fractional change"""
    return x / x.isel(year=0) - 1


def check_dims(x):
    """make sure dimensions are ok before averaging"""
    ## check if latitude is in ssh
    if "latitude" in x.dims:
        x_ = copy.deepcopy(x)
    else:
        x_ = x.expand_dims("latitude")

    ## check if z_t is in ssh
    if "z_t" in x.dims:
        x_ = copy.deepcopy(x_)
    else:
        x_ = x_.expand_dims("z_t")

    return x_


def get_ddt(data):
    """differentiate with respect to time"""
    data_ = copy.deepcopy(data)
    data_ = data_.assign_coords({"t_idx": ("time", np.arange(len(data_.time)))})
    data_ = data_.swap_dims({"time": "t_idx"})
    return data_.differentiate("t_idx").swap_dims({"t_idx": "time"})

### Plotting

In [ ]:
def make_scatter_ax(ax, anom_, xvar, yvar, month, label, by_season=True):
    """scatter plot of data for given month"""

    ## prep func
    if by_season:
        get_season = lambda x: src.utils.sel_month(
            x.resample({"time": "QS-JAN"}).mean(), month
        )

    else:
        get_season = lambda x: src.utils.sel_month(x, month)

    prep = lambda x: get_season(x).transpose("time", "member")

    ## get plot data
    plot_data = (prep(anom_[xvar]), prep(anom_[yvar]))

    ## get stats
    corr = xr.corr(*plot_data)
    cov = xr.cov(*plot_data)
    m = cov / plot_data[0].var()

    ## plot data
    ax.scatter(*plot_data, s=3, label=f"m = {m.item():.1e}\nr = {corr.item():.2f}")
    ax.set_title(f"{label}")

    ## formatting
    ax_kwargs = dict(ls="--", c="gray", lw=0.5)
    ax.axvline(0, **ax_kwargs)
    ax.axhline(0, **ax_kwargs)
    ax.legend(prop=dict(size=10))

    return ax


def make_scatter(anom_, xvar, yvar, month, by_season=True):
    """scatter plot of data for given month"""

    fig, axs = plt.subplots(1, 4, figsize=(11, 2.5), layout="constrained")

    for ax, t_idx, label in zip(
        axs,
        [["1850", "1879"], ["1995", "2024"], ["2035", "2064"], ["2071", "2100"]],
        ["1865", "2010", "2050", "2085"],
    ):

        ## helper func
        prep = lambda x: x.sel(time=slice(*t_idx))

        ## scatter plot of data
        ax = make_scatter_ax(
            ax,
            prep(anom_),
            xvar=xvar,
            yvar=yvar,
            month=month,
            by_season=by_season,
            label=label,
        )

    ## format/scale axes
    src.utils.set_lims(axs)
    for ax in axs[1:]:
        ax.set_yticks([])

    return fig, axs


def plot_timeseries(coefs, sel_fn=lambda x: x):
    """plot timeseries comparison"""

    fig, axs = plt.subplots(1, 3, figsize=(8, 2.5))

    ## loop thru pos and negative
    for i, (name, color) in enumerate(zip(["pos", "neg"], ["r", "b"])):

        ## plot median and bounds
        for q, lw in zip([0.5, 0.1, 0.9], [2, 0.6, 0.6]):

            ## plot neutral and pos/or neg
            for name_, color_ in zip(["all", name], ["k", color]):

                ## finally, plot data
                axs[i].plot(
                    coefs.year,
                    sel_fn(coefs)[name_].quantile(q=q, dim="member"),
                    c=color_,
                    lw=lw,
                )

            ## plot on shared axs
            axs[2].plot(
                coefs.year,
                sel_fn(coefs)[name].quantile(q=q, dim="member"),
                c=color,
                lw=lw,
            )

    ## formatting
    for ax in axs:
        ax.set_xticks([1870, 2010, 2080])
        ax_kwargs = dict(ls="--", c="gray", lw=0.8)
        ax.axvline(2010, **ax_kwargs)
        ax.axhline(0, **ax_kwargs)
    src.utils.set_lims(axs)

    for ax in axs[1:]:
        ax.set_yticks([])

    return fig, axs


def plot_zonal_structure(coefs, sel_fn=lambda x: x):
    """plot zonal structure of coefficients over time"""

    fig, axs = plt.subplots(1, 4, figsize=(10, 2.5), layout="constrained")

    for ax, y in zip(axs, [1865, 2010, 2050, 2085]):

        ## get data for year
        ax.set_title(y)
        coefs_y = sel_fn(coefs).sel(year=y, method="nearest")

        ## select data
        for n, color in zip(["all", "pos", "neg"], ["k", "r", "b"]):

            ax.plot(coefs.longitude, coefs_y[n].mean("member"), c=color)

    src.utils.set_lims(axs)
    for ax in axs[1:]:
        ax.set_yticks([])

    for ax in axs:
        ax.set_xticks([140, 190, 240, 280])
        ax_kwargs = dict(ls="--", c="gray", lw=0.8)
        ax.axhline(0, **ax_kwargs)

    return fig, axs


def add_vticks(axs, xticks, xlines=None):
    """add vertical lines to axs"""

    ## specify line style
    ax_kwargs = dict(ls="--", c="gray", lw=0.8)

    ## loop thru axs
    for ax in axs:
        ax.set_xticks(xticks)
        if xlines is not None:
            for x0 in xlines:
                ax.axvline(x0, **ax_kwargs)

    return


def plot_vertical_structure(coefs, sel_fn):
    """plot vertical structure of coefficients over time"""

    fig, axs = plt.subplots(1, 4, figsize=(10, 2.5), layout="constrained")

    for ax, y in zip(axs, [1865, 2010, 2050, 2085]):

        ## get data for year
        ax.set_title(y)
        coefs_y = sel_fn(coefs).sel(year=y, method="nearest")

        ## select data
        for n, color in zip(["all", "pos", "neg"], ["k", "r", "b"]):

            ax.plot(coefs_y[n].mean("member"), coefs.z_t, c=color)

    for ax in axs:
        ax.set_ylim([150, 5])

        ax_kwargs = dict(ls="--", c="gray", lw=0.8)
        ax.axhline(50, **ax_kwargs)
        ax.axhline(80, **ax_kwargs)
        ax.axvline(0, **ax_kwargs)

    src.utils.set_lims(axs)
    axs[0].set_yticks([0, 50, 80])
    for ax in axs[1:]:
        ax.set_yticks([])
    return fig, axs

## Load data

#### $T$, $h$

In [ ]:
## open data
Th = src.utils.load_cesm_indices(load_z20=True)

### Spatial data

#### most data

In [ ]:
## load spatial data
forced, anom = src.utils.load_consolidated()

## add T,h information
anom = xr.merge([anom, Th])

#### "wide" subsurface data

In [ ]:
## should we use "wide" data?
USE_WIDE = True

## load spatial data
forced_wide, anom_wide = load_consolidated_wide()

if USE_WIDE:

    for v in list(forced_wide):
        forced[v] = forced_wide[v]
        anom[v] = anom_wide[v]

#### max grad thermocline

In [ ]:
h_mg_forced, h_mg = src.utils.load_h_data(max_grad=True)

### scaling for mean thermocline depth

#### Mean thermocline depth

In [ ]:
hbar_scale = xr.open_dataarray(
    pathlib.Path(SAVE_FP, "cesm_Hbar_scale_v2.nc"),
)

### Compute OHC

In [ ]:
## should we use mixed layer T?
# use_T_ml = True
use_T_ml = False

## specify subsetting funcs
LATS = dict(latitude=slice(-5, 5))
LATS_H = dict(latitude=slice(-5, 5))
LONS_E = dict(longitude=slice(210, 270))
# LONS_W = dict(longitude=slice(120, 160))
LONS_W = dict(longitude=slice(120, 180))
LONS_TAU = dict(longitude=slice(150, 230))

Set funcs

In [ ]:
## helper func to select and avg
def sel_helper(x, lats, lons):
    """helper func to avg over lats/lons"""

    ## first, average over lons
    x_avg = x.sel(lons).mean("longitude")

    if "latitude" in x_avg.dims:
        x_avg = x_avg.sel(lats).mean("latitude")

    return x_avg


## specify functions
TAU_FN = lambda x: sel_helper(x, LATS, LONS_TAU)
TAU_FN_3 = lambda x: sel_helper(x, LATS, LONS_E)
He_FN = lambda x: sel_helper(x, LATS_H, LONS_E)
Hw_FN = lambda x: sel_helper(x, LATS_H, LONS_W)
Hgrad_FN = lambda x: He_FN(x) - Hw_FN(x)


## specify entrainment / ML averages
LON_AVG = lambda x: x.sel(longitude=slice(210, 270)).mean("longitude")
LON_AVG_34 = lambda x: x.sel(longitude=slice(190, 240)).mean("longitude")
ENT_AVG = lambda x: x.sel(z_t=slice(50, 80)).mean("z_t")
ML_AVG = lambda x: x.sel(z_t=slice(None, 50)).mean("z_t")

## get T3 volume avg
T3_ENT_AVG = lambda x: ENT_AVG(LON_AVG(x))
T3_ML_AVG = lambda x: ML_AVG(LON_AVG(x))
T34_ML_AVG = lambda x: ML_AVG(LON_AVG_34(x))

if use_T_ml:
    anom["T_3"] = src.utils.reconstruct_wrapper(
        anom[["T", "T_comp"]],
        fn=T3_ML_AVG,
    )["T"]

In [ ]:
## should we use OHC?
USE_OHC = True

lon_avg = lambda x, lons: x.sel(lons).mean("longitude")
lat_avg = lambda x: x.sel(latitude=slice(-5, 5)).mean("latitude")

if USE_OHC:

    ## compute ohc
    anom["h_w"] = src.utils.reconstruct_wrapper(
        anom_wide[["T", "T_comp"]],
        lambda x: lon_avg(x.integrate("z_t"), LONS_W) / 300,
    )["T"]
    anom["h_e"] = src.utils.reconstruct_wrapper(
        anom_wide[["T", "T_comp"]],
        lambda x: lon_avg(x.integrate("z_t"), LONS_E) / 300,
    )["T"]

else:
    anom["h_w"] = src.utils.reconstruct_wrapper(
        anom[["ssh", "ssh_comp"]],
        lambda x: lat_avg(lon_avg(x, LONS_W)),
    )["ssh"]
    anom["h_e"] = src.utils.reconstruct_wrapper(
        anom[["ssh", "ssh_comp"]],
        lambda x: lat_avg(lon_avg(x, LONS_E)),
    )["ssh"]

## Estimate $R$

#### Get data

In [ ]:
## get data (remove median as well)
x = xr.merge([Th[T_VAR], anom["h_w"]])
# x /= x.std()
x = window(x, subtract_median=SUBTRACT_MEDIAN)
y = get_ddt(x).rename({f"{v}":f"ddt_{v}" for v in list(x)})
z = xr.merge([x,y])

#### Fit

In [ ]:
import warnings

def get_fits_over_time(data_rolling, model, by_member=False, **fit_kwargs):
    """Get RO fits for each ensemble member as a function of time."""

    ## empty list to hold results
    fits = []

    ## loop through years
    for y in tqdm.tqdm(data_rolling.year):

        ## get data for year
        data_y = data_rolling.sel(year=y)

        if by_member:

            ## separate fit for each ensemble member
            fits_ = []
            for m in data_rolling.member:
                with warnings.catch_warnings(action="ignore"):
                    fits_.append(model.fit_matrix(data_y.sel(member=m), **fit_kwargs))
            fit = xr.concat(fits_, dim=data_y.member)

        else:

            ## fit for all ensemble members together
            with warnings.catch_warnings(action="ignore"):
                fit = model.fit_matrix(data_y, **fit_kwargs)

        ## track fits
        fits.append(fit.drop_vars(["time", "X", "Y", "Yfit"]))

    ## put back in xarray
    fits = xr.concat(fits, dim=data_rolling.year)

    return fits

In [ ]:
## parameters for fitting
MODEL = src.XRO.XRO(ncycle=12, ac_order=4, is_forward=True)
fit_kwargs = dict(ac_mask_idx=None, maskNT=["T2", "T3"], maskNH=[])

## get fits
fits = get_fits_over_time(
    x,
    model=MODEL,
    by_member=False,
    **fit_kwargs,
)

## extract parameters
params = src.utils.get_params(fits=fits, model=MODEL)

## get change from initial period
delta_params = params - params.isel(year=0)

In [ ]:
## get coefficients
coefs_nl = fits["NROT_Lac"].sel(nro_form=["T2", "T3"])
R3 = fits["Lac"].isel(ranky=0, rankx=0).expand_dims(nro_form=["T"])
eps3 = -fits["Lac"].isel(ranky=1, rankx=1)
coefs3 = xr.concat([R3, coefs_nl], dim="nro_form").rename({"nro_form": "form"})

## get coefficient array
# T0 = 1.5
sigma = x[T_VAR].std().values.item()
w = sigma * np.linspace(-3, 3)
W = np.stack([w**i for i in range(3)], axis=1)
W = xr.DataArray(W, coords=dict(T=w, form=coefs3.form), dims=["T", "form"])

## reconstruct nonlinear R
R_nl = (coefs3 * W).sum("form")
R_nl_pos = R_nl.where(R_nl["T"]>0).mean("T")
R_nl_neg = R_nl.where(R_nl["T"]<0).mean("T")


# T0 = sigma * 1.5
T0 = 1.5
R_nl_pos2 = R_nl.sel(T=T0, method="nearest")
R_nl_neg2 = R_nl.sel(T=-T0, method="nearest")

\begin{align}
    R &= R_0 + R_1T + R_2 T^2
\end{align}

In [ ]:
sel = lambda x : x.mean('cycle')#.differentiate("year")
fig,ax = plt.subplots(figsize=(3,2.5))
ax.plot(params.year, .2*sel(params["R"]), c="k")
ax.plot(coefs_nl.year, sel(coefs_nl.sel(nro_form="T2")))
ax.plot(coefs_nl.year, sel(coefs_nl.sel(nro_form="T3")))
ax.axhline(0, ls="--", lw=.8, c="gray")
ax.axvline(2050, ls="--", lw=.8, c="gray")
plt.show()

In [ ]:
fig,ax = plt.subplots(figsize=(3,2.5), layout="constrained")

ax.plot(params.year, R_nl.mean(["T","cycle"]), c="k")
ax.plot(params.year, R_nl_pos.mean("cycle"), c="r")
ax.plot(params.year, R_nl_neg.mean("cycle"),  c="b")

ax.plot(params.year, params["R"].mean("cycle"), c="k", ls="--")
ax.plot(params.year, R_nl_pos2.mean("cycle"), c="r", ls="--")
ax.plot(params.year, R_nl_neg2.mean("cycle"),  c="b", ls="--")

add_vticks([ax], xticks=[1870,2010,2090], xlines=[2010,2050])
plt.show()

## Estimate $\alpha$

### Load data

#### NHF in Niño 3 region

In [ ]:
## get scaling factor to convert to damping rate (units of K/mo)
sec_per_mo = 8.64e4 * 30
sec_per_yr = 12 * sec_per_mo
rho = 1.02e3
Cp = 4.2e3
H0 = 50
SCALE_FAC = sec_per_yr / (rho * Cp * H0)

Q_3 = SCALE_FAC * src.utils.reconstruct_wrapper(
    anom[["nhf","nhf_comp"]], fn=src.utils.get_nino3,
).rename({"nhf":"Q"})

#### merge data

In [ ]:
## get data (remove median as well)
XY = xr.merge([Th[T_VAR], anom["h_w"], Q_3])
XY = window(XY, subtract_median=SUBTRACT_MEDIAN)

### Fit nonlinear alpha

In [ ]:
def regress_pinv(data, x_var, y_var, order=4):
    """do nonlinear regression"""

    ## get feature matrix
    # X_nl = xr.merge([(data[x_var]**i).rename(f"{x_var}**{i}") for i in range(order)])
    X_nl = xr.merge([(data[x_var]**i).rename(i) for i in range(order)])

    ## prep data
    prep = lambda x : x.transpose("member",...)
    X_ = prep(X_nl.to_dataarray(dim="order"))
    Y_ = prep(data[y_var])

    ## empty array to hold results
    m = xr.DataArray(
        coords={"order":X_.order.values, "member":X_.member.values}, dims=["member","order"]
    )
    
    ## do regression
    X_pinv = np.linalg.pinv(X_.values)
    m.values = np.einsum("mi,mij->mj", Y_.values, X_pinv)

    return m

In [ ]:
## subset and fit
kwargs = dict(x_var=T_VAR, y_var="Q", order=3)
alpha = []
for y in tqdm.tqdm(XY.year[:-1]):
    alpha.append(XY.sel(year=y).groupby("time.month").map(regress_pinv, **kwargs))
alpha = xr.concat(alpha, dim=XY.year[:-1])

## feature mat
zz = np.linspace(-3,4)
ZZ = np.stack([zz**i for i in alpha.order.values], axis=1)
ZZ = xr.DataArray(
    ZZ, coords=dict(order=alpha.order.values, T=zz), dims=["T","order"]
)

## reconstruct
Q_recon = (ZZ * alpha).sum("order")
Q_recon_ = Q_recon.mean("member")

### Plot data

In [ ]:
m_idx=10

sel_mo = lambda x : x.isel(time=slice(m_idx,None,12))
fig,axs = plt.subplots(1,3,figsize=(9,3))

for ax, y in zip(axs, [1870, 2010, 2080]):

    sel = lambda x : sel_mo(x).sel(year=y)
    
    ax.scatter(
        sel(XY[T_VAR]), sel(XY["Q"]), s=2,
    )
    ax_kwargs = dict(lw=.8, ls="--", c="gray")
    ax.axhline(0, **ax_kwargs)
    ax.axvline(0, **ax_kwargs)
    ax.axvline(-2, **ax_kwargs)
    ax.axvline(2, **ax_kwargs)

    ax.plot(Q_recon["T"], Q_recon_.sel(month=m_idx+1, year=y), c="k")

src.utils.set_lims(axs)
for ax in axs[1:]:
    ax.set_yticks([])
plt.show()

## Compute $R_o$

In [ ]:
## make sure dims match
alpha_ = alpha.mean("member").isel(order=slice(1,None))
zeros = xr.zeros_like(alpha_.isel(order=1)).expand_dims(order=[3])
alpha_ = xr.concat([alpha_,zeros], dim="order")

In [ ]:
coefs_ = coefs3.rename({"form":"order","cycle":"month"})
coefs_ = coefs_.assign_coords({"month":alpha_.month, "order":alpha_.order})
coefs_ = coefs_.sel(year=alpha_.year)

## decompose
R_ = xr.merge([coefs_.rename("R"), alpha_.rename("Ra")])
R_["Ro"] = R_["R"]-R_["Ra"]

In [ ]:
## how should we scale the feature matrix?
## One of {"constant", "by_month", "sliding_by_month"}
SCALE_TYPE = "by_month"

## get coefficient array (standardized units)
z = np.arange(-3, 3.1, 0.1)
z = xr.DataArray(z, coords=dict(sigma=z))

## get scale
sigma_T = XY[T_VAR].groupby("time.month").std(["time", "member"])

## average as needed
if SCALE_TYPE == "constant":
    sigma_T = sigma_T.mean(["year", "month"]) * xr.ones_like(sigma_T)

elif SCALE_TYPE == "by_month":
    sigma_T = sigma_T.mean("year") * xr.ones_like(sigma_T)

## scale feature mat by sigma_T
Z = sigma_T * z.expand_dims({"year": R_.year, "month": R_.month})

## expand to get coefficients
Z = xr.concat([Z**i for i in range(3)], dim=R_.order)

## check everything makes sense
print(np.allclose(Z.sel(order=2).sel(sigma=1, method="nearest"), sigma_T.sel(year=Z.year)))

## reconstruct nonlinear R
R_recon = (R_ * Z).sum("order")

## Plot results

In [ ]:
## specify amplitude for nonlinearity to plot (note: could just look at quadratic coef...)
AMP = 1.5

## specify months to plot
# PLOT_MONTHS = slice(9,12)
PLOT_MONTHS = slice(1,12)
# PLOT_MONTHS = [11,12,1,2]

## normalize
norm = lambda x : x-x.isel(year=0)
# norm = lambda x : x

####### plot ocean/atmos. over time

## get plot data
R_plot = norm(R_recon.sel(month=PLOT_MONTHS).mean("month"))

fig,axs = plt.subplots(1,3,figsize=(9,2.5), layout="constrained")

for ax, n, label in zip(axs, list(R_),[r"$R$",r"$R_a$", r"$R_o$"]):

    ## modify sign of damping (optional)
    s = 1 if (n=="Ra") else 1
    
    ## plot data
    for sc, c in zip([-AMP, 0, AMP], ["b","k","r"]):
        ax.plot(R_recon.year, s*R_plot[n].sel(sigma=sc, method="nearest"), c=c)

    ## label/formatting
    ax.set_title(label)
    ax.set_yticks([])
    ax.axhline(0, ls="--", c="gray", lw=.8)

## label/formatting
add_vticks(axs, xticks=[1870, 2010, 2080], xlines=[2010, 2050])
src.utils.set_lims(axs)
axs[0].set_yticks([-.5,0,0.5])

plt.show()


####### Plot symmetric/asymmetric changes

## normalize data
R_plot = norm(R_.sel(month=PLOT_MONTHS).mean("month"))

## get scale to normalize plots
SCALE = AMP * sigma_T.sel(month=PLOT_MONTHS).mean(["year","month"]).values.item()

## set up plot
fig,axs = plt.subplots(1,3,figsize=(9,2.5), layout="constrained")

for ax, order in zip(axs, R_plot.order.values):

    ## specify colors to use
    colors = ["k"]+sns.color_palette()[:2]

    ## get scaling factor
    R_scaled_plot = R_plot.sel(order=order) * (SCALE ** (order-1))

    ## plot data
    for n, c, l in zip(["R","Ro","Ra"], colors, [r"$R$", r"$R_o$", r"$R_a$"]):
        ax.plot(R_plot.year, R_scaled_plot[n], label=l, c=c)

    ## label/formatting
    ax.set_title(f"Order {order}")
    ax.set_yticks([])
    ax.axhline(0, ls="--", c="gray", lw=.8)

## label/formatting
add_vticks(axs, xticks=[1870, 2010, 2080], xlines=[2010, 2050])
src.utils.set_lims(axs)
axs[0].set_yticks([-.5,0,0.5])
axs[0].set_ylabel(r"yr$^{-1}$")
axs[1].legend()

plt.show()

## Scratch

### Seasonal dependence (harmonics)

In [ ]:
## specify order
ORDER = 2

## make feature mat
f = np.fft.fftfreq(n=12, d=1/12)
months = np.arange(1,13)
theta = (months-1) * 2*np.pi/12
# theta = np.linspace(0,2*np.pi)

arg = f[:,None] * theta[None,:]
F = np.exp(1j * arg)
F_sc = np.stack([np.cos(arg), np.sin(arg)], axis=-1)
F_st = np.stack([F.real, F.imag], axis=-1)

## restrict feature mat
is_valid_order = np.abs(f)<=ORDER
F = F[is_valid_order]
F_sc = F_sc[is_valid_order]
F_st = F_st[is_valid_order]

## check it works
print(np.allclose(F, F_sc[...,0] + 1j*F_sc[...,-1]))
print(np.allclose(F, F_st[...,0] + 1j*F_st[...,-1]))

In [ ]:
## generate random coefficients
rng = np.random.default_rng()
fhat = rng.normal(size=(ORDER+1)) + 1j * rng.normal(size=(ORDER+1))
fhat[0] = fhat[0].real
fhat = np.concatenate([fhat, fhat[-1:-ORDER-1:-1]])

## reconstruct
res = fhat[:,None] * F
res2 = fhat.real[:,None] * F_st[...,0] + fhat.imag[:,None] * F_st[...,-1]

## check it 
print(np.allclose(res[[0,1,-1]].sum(0).real, res2[[0,1,-1]].sum(0)))

## plot
fig,ax = plt.subplots(figsize=(3,2.5))
ax.plot(theta, res.sum(0).real)
ax.plot(theta, res[[0]].sum(0).real)
ax.plot(theta, res[[0,1,-1]].sum(0).real)
ax.plot(theta, res2.sum(0).real)
plt.show()

### regression struff

In [ ]:
def reshape_features(x):
    """reshape data for regression"""

    ## stack sample dim
    x = x.stack(sample=["member","time"])

    ## put variables in dataarray and get feature dim
    x = x.to_dataarray(dim="varname")
    x = x.stack(feature=["varname", "order"])
    
    return x.transpose("feature", "sample")

def regress(X, Y):
    """regression of X onto Y"""

    ## reshape data
    X_ = reshape(X)
    Y_ = reshape(Y)

    ## empty array to hold result
    return Y_.values @ np.linalg.pinv(X_.values)
    